# Bölüm klipleri: Wan 2.2 (5B) ile daha canlı hareket (ücretsiz)

`kare-uret` çıktısındaki başlangıç karelerinden **Wan 2.2 TI2V-5B** ile klip üretir.
LTX'e göre insan hareketi ve yüz tutarlılığı daha iyidir, ama T4'te daha yavaştır
(~700 sn / 3,4 sn'lik parça; 3 dakikalık bölüm ~11 saat).

**Ayarlar:** Accelerator **GPU T4 x2**, Internet **On**, **Add Input** → `kare-uret` not defterinin çıktısı
(keyframes + plan.json). Sonra **Run All**. Bitince Output'ta `klipler-wan-*.zip` olur.

Kaggle oturumu 12 saatle sınırlı: bölümü ikiye bölün, ör. bir kopyada `ONLY = [f"c{i:02d}" for i in range(1, 11)]`,
diğerinde `range(11, 21)`.

In [ ]:
# 1) Kurulum
!pip uninstall -y -q torchao
!pip install -q -U "diffusers>=0.35" "transformers>=4.48" accelerate ftfy sentencepiece imageio imageio-ffmpeg

In [ ]:
# 2) Ayarlar
import glob, json, os, time, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # CUDA başlamadan önce
import torch

ONLY = None           # None = tüm sahneler
# (genişlik, yükseklik, parça başına kare 4k+1)
SIZES = [(832, 480, 81)]   # iki T4'te sığdığı doğrulandı (~700 sn / parça)
FPS = 24
STEPS = 30
GUIDANCE = 5.0
SEED = 23
DTYPE = torch.float16        # T4 bfloat16'yı donanımda desteklemez; siyah kare çıkarsa torch.bfloat16 deneyin

plans = glob.glob("/kaggle/input/**/plan.json", recursive=True)
assert plans, "plan.json bulunamadı. 'Add Input' ile kare-uret çıktısını ekleyin."
plan = json.load(open(plans[0]))
firsts = glob.glob(f"/kaggle/input/**/keyframes/{plan['clips'][0]['key']}.png", recursive=True)
assert firsts, "Kareler bulunamadı."
KEY_DIR = os.path.dirname(firsts[0])
OUT_DIR = "/kaggle/working/clips"
os.makedirs(OUT_DIR, exist_ok=True)
print(plan["series"], "·", len(plan["clips"]), "sahne ·", KEY_DIR)

In [ ]:
# 3) Modeli yükle: iki T4'e bölünür (metin kodlayıcı cuda:1, video modeli + VAE cuda:0).
# Tek kartta model_cpu_offload ve group offload ya belleğe ya da 29 GB RAM'e sığmadı.
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline, WanTransformer3DModel
from diffusers.utils import export_to_video, load_image
from transformers import AutoTokenizer, UMT5EncoderModel

MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
assert torch.cuda.device_count() >= 2, "Accelerator olarak GPU T4 x2 seçin."
G0, G1 = torch.device("cuda:0"), torch.device("cuda:1")
if "pipe" not in globals():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
    te = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=DTYPE).to(G1).eval()
    transformer = WanTransformer3DModel.from_pretrained(MODEL_ID, subfolder="transformer", torch_dtype=DTYPE).to(G0)
    vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32).to(G0)
    pipe = WanImageToVideoPipeline.from_pretrained(
        MODEL_ID, transformer=transformer, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=DTYPE)
    if hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()

@torch.inference_mode()
def encode(text, max_len=512):
    # diffusers'ın Wan metin kodlamasıyla aynı: dolgu kısmı sıfırlanır
    t = tok([text], padding="max_length", max_length=max_len, truncation=True,
            add_special_tokens=True, return_attention_mask=True, return_tensors="pt")
    ids, mask = t.input_ids.to(G1), t.attention_mask.to(G1)
    n = int(mask.sum())
    e = te(ids, mask).last_hidden_state[0, :n]
    e = torch.cat([e, e.new_zeros(max_len - n, e.size(1))])[None]
    if not torch.isfinite(e).all():
        print("⚠ metin kodlamasında NaN/inf (fp16 taşması)")
    return e.to(G0, DTYPE)

print("Hazır:", [torch.cuda.get_device_name(i) for i in range(2)],
      "· GPU0", f"{torch.cuda.memory_allocated(0) / 2**30:.1f} GB · GPU1", f"{torch.cuda.memory_allocated(1) / 2**30:.1f} GB")

In [ ]:
# 4) Klipleri üret: Wan bir seferde ~3,4 sn üretir; uzun sahnelerde her parça bir öncekinin son karesinden devam eder
import gc, math, psutil
NEGATIVE = ("static, frozen, blurry, distorted face, deformed eyes, extra fingers, bad hands, "
            "jitter, flicker, watermark, text, low quality, cartoon, character leaving frame, back to camera, "
            "extra people, sudden cut")

def run(image, pe, ne):
    return pipe(
        image=image.resize((W, H)), prompt_embeds=pe, negative_prompt_embeds=ne,
        height=H, width=W, num_frames=N,
        num_inference_steps=STEPS, guidance_scale=GUIDANCE,
        generator=torch.Generator("cpu").manual_seed(SEED),
    ).frames[0]

W, H, N = SIZES[0]
ne = encode(NEGATIVE)
done, failed = [], []
for c in plan["clips"]:
    key = c["key"]
    if ONLY and key not in ONLY:
        continue
    out = os.path.join(OUT_DIR, f"{key}.mp4")
    if os.path.exists(out):
        print(f"✔ {key} zaten var"); continue
    t0 = time.time()
    try:
        pe = encode(f"{c['motion']}, natural lifelike human movement, cinematic camera. {c['keyframe']}. {plan['style']}")
        need = int(c["duration"] * FPS)
        segs = math.ceil((need - 1) / (N - 1))
        image, frames = load_image(os.path.join(KEY_DIR, f"{key}.png")), []
        for k in range(segs):
            part = run(image, pe, ne)
            frames += part if not frames else part[1:]   # yeni parçanın ilk karesi öncekinin son karesi
            image = part[-1]
            gc.collect(); torch.cuda.empty_cache()
        export_to_video(frames[:need], out, fps=FPS)
        done.append(key)
        print(f"✔ {key}  {segs} parça · {len(frames[:need]) / FPS:.1f} sn · {time.time() - t0:.0f} sn · "
              f"GPU tepe {torch.cuda.max_memory_allocated(0) / 2**30:.1f} GB · RAM {psutil.virtual_memory().used / 2**30:.1f} GB")
    except Exception as e:
        failed.append(key)
        print(f"✖ {key}: {type(e).__name__}: {e}")
    gc.collect(); torch.cuda.empty_cache()
print(f"Biten: {len(done)} · Hatalı: {failed or 'yok'}")

In [ ]:
# 5) Zip'le (bölünmüş çalıştırmada her kopya kendi sahnelerini zip'ler; Output'ta yalnızca zip kalır)
part = f"-{ONLY[0]}-{ONLY[-1]}" if ONLY else ""
zip_base = f"/kaggle/working/klipler-wan-{plan['series']}-e{plan['episode']}{part}"
shutil.make_archive(zip_base, "zip", OUT_DIR)
shutil.rmtree(OUT_DIR, ignore_errors=True)
print("🎬 İndirilecek dosya:", zip_base + ".zip")